# Week 08 — Home exercise 1: Predict the output

**Solution proposal.**

For each snippet: what it displays, and why. Then the two broken snippets.

Every snippet here runs without an error. That is the point of the set — the mistakes this week
produces are not the kind that stop your program.

In [1]:
import numpy as np
import pandas as pd

## a) What a `groupby` gives back

Displays `<class 'pandas.Series'>`, then a two-row Series labeled `A` and `B` holding 40 and 60, then
`40`.

Two things worth noticing. It is a **Series**, not a DataFrame — one column of answers. And `shop` is
not a column of it: the shop names are the **index**, which is why `result["A"]` works and
`result["shop"]` would raise a `KeyError`.

`reset_index()` turns it back into an ordinary two-column table when you need one.

In [2]:
sales = pd.DataFrame({
    "shop": ["A", "B", "A", "B"],
    "day":  [1, 1, 2, 2],
    "sold": [10, 20, 30, 40],
})

result = sales.groupby("shop")["sold"].sum()

print(type(result))
print(result)
print(result["A"])

<class 'pandas.Series'>
shop
A    40
B    60
Name: sold, dtype: int64
40


## b) Shape is the whole difference

Displays `(2,)` then `(4,)`.

`.mean()` **collapses**: one row per group. `.transform("mean")` **preserves shape**: one row per
original row, each holding its own group's answer.

That is why only the second can be assigned to a column of `sales`. Assigning the first would ask
pandas to line up two labels — `A` and `B` — against four row numbers, find no match anywhere, and
fill the column with `NaN`.

In [3]:
print(sales.groupby("shop")["sold"].mean().shape)
print(sales.groupby("shop")["sold"].transform("mean").shape)

sales["group_mean"] = sales.groupby("shop")["sold"].transform("mean")
print(sales)

(2,)
(4,)
  shop  day  sold  group_mean
0    A    1    10        20.0
1    B    1    20        30.0
2    A    2    30        20.0
3    B    2    40        30.0


## c) The rows that quietly are not there

Displays `blue 20`, `red 50` — and then `100`.

The group totals come to 70. The column comes to 100. **The 30 hours belonging to the row with no
team have been dropped**, because `groupby` has nowhere to put a row whose key is missing.

No error, no warning, and a report that is 30% short. `dropna=False` keeps them in a group of their
own.

In [4]:
staff = pd.DataFrame({
    "team":  ["red", "blue", None, "red"],
    "hours": [10, 20, 30, 40],
})

print(staff.groupby("team")["hours"].sum())
print("sum of the groups:", staff.groupby("team")["hours"].sum().sum())
print("sum of the column:", staff["hours"].sum())
print()
print(staff.groupby("team", dropna=False)["hours"].sum())

team
blue    20
red     50
Name: hours, dtype: int64
sum of the groups: 70
sum of the column: 100

team
blue    20
red     50
NaN     30
Name: hours, dtype: int64


## d) A left join that grew

Displays `2 -> 3`, and a table where `x` appears twice.

A **left** join sounds like a promise to keep the left table's shape, and it is not one. It promises
to keep every left row *at least* once; a key that matches twice on the right produces two rows.

Here `x` is in `right` twice, so it comes out twice — carrying `v = 1` into both. If you now averaged
`v`, `x` would count double.

In [5]:
left = pd.DataFrame({"k": ["x", "y"], "v": [1, 2]})
right = pd.DataFrame({"k": ["x", "x", "y"], "w": [10, 20, 30]})

merged = left.merge(right, on="k", how="left")

print(len(left), "->", len(merged))
print(merged)

2 -> 3
   k  v   w
0  x  1  10
1  x  1  20
2  y  2  30


`validate="one_to_one"` would have refused to run it.

In [6]:
try:
    left.merge(right, on="k", how="left", validate="one_to_one")
except Exception as error:
    print(type(error).__name__, "-", str(error).split("\n")[0])

MergeError - Merge keys are not unique in right dataset; not a one-to-one merge


## e) A change that crosses a border

Displays a `change` column of `NaN, 1.0, 98.0, 1.0`.

The **98** is the bug. It is country B's first year minus country A's last year — two different
countries, subtracted from each other because they happen to be adjacent rows.

The right answer is `NaN` there: B has no earlier year in this table. `.diff()` counts rows, and it
has no idea that row 2 belongs to somebody else.

In [7]:
panel = pd.DataFrame({
    "country": ["A", "A", "B", "B"],
    "year":    [2020, 2021, 2020, 2021],
    "value":   [1.0, 2.0, 100.0, 101.0],
})

panel["change"] = panel["value"].diff()
panel["change_grouped"] = panel.groupby("country")["value"].diff()

print(panel)

  country  year  value  change  change_grouped
0       A  2020    1.0     NaN             NaN
1       A  2021    2.0     1.0             1.0
2       B  2020  100.0    98.0             NaN
3       B  2021  101.0     1.0             1.0


## f) Dates that are still text

The sort displays the rows in the order `02/11/2020`, `05/06/2021`, `10/01/2021` — which is
**alphabetical by day**, not chronological. `"02/..."` sorts before `"05/..."` sorts before
`"10/..."`, and the year at the end of the string never gets looked at.

Then `pd.to_datetime` reads them month-first, in the American style, and turns `10/01/2021` into
**1 October 2021** rather than 10 January.

Both operations succeeded. Both are wrong. Sorting text that looks like a date only works when the
date is written year-first, and conversion only works when you say what the format is.

In [8]:
dates = pd.DataFrame({
    "date":  ["10/01/2021", "02/11/2020", "05/06/2021"],
    "value": [1, 2, 3],
})

print(dates.sort_values("date"))
print()
print("guessed:", list(pd.to_datetime(dates["date"]).dt.date))
print("told:   ", list(pd.to_datetime(dates["date"], format="%d/%m/%Y").dt.date))

         date  value
1  02/11/2020      2
2  05/06/2021      3
0  10/01/2021      1

guessed: [datetime.date(2021, 10, 1), datetime.date(2020, 2, 11), datetime.date(2021, 5, 6)]
told:    [datetime.date(2021, 1, 10), datetime.date(2020, 11, 2), datetime.date(2021, 6, 5)]


## Broken snippet 1: the join threw two regions away

Two separate faults, and they compound.

The merge is on `name`, and two entity names in the lookup file carry a **trailing space**. With
`how="inner"`, the 48 rows that fail to match are deleted without comment.

Then `groupby("region")` is asked to total the survivors — and had the join been a left join instead,
those 48 rows would still have been dropped at that step, because their `region` would have been
missing. Either way the number comes out short and nothing says so.

Merging on `code` fixes both, because `code` is clean on both sides.

In [9]:
co2 = pd.read_csv("../data/co2_emissions.csv")
info = pd.read_csv("../data/country_info.csv")

broken = co2.merge(info, left_on="country", right_on="name", how="inner")
fixed = co2.merge(info[["code", "region"]], on="code", how="left", validate="many_to_one")

print("rows, broken:", len(broken))
print("rows, fixed: ", len(fixed))
print()
print("total by region, broken:", broken.groupby("region")["co2_total"].sum().sum().round(1))
print("total by region, fixed: ", fixed.groupby("region")["co2_total"].sum().sum().round(1))
print("the column itself:      ", co2["co2_total"].sum().round(1))

rows, broken: 6192
rows, fixed:  6240

total by region, broken: 6221626.7
total by region, fixed:  6279444.6
the column itself:       6279444.6


## Broken snippet 2: `.diff()` without a group

Sorting by country and year makes the table *look* like a time series, so `.diff()` runs happily —
and every country's first row gets the difference against the previous country's last row.

Norway's own numbers are fine, because Norway's first row is the only one affected and you would have
to scroll to it to notice. That is exactly what makes this dangerous: the visible rows are right.

In [10]:
co2 = pd.read_csv("../data/co2_emissions.csv").sort_values(["country", "year"])

co2["change_wrong"] = co2["co2_total"].diff()
co2["change"] = co2.groupby("country")["co2_total"].diff()

wrong_rows = co2["change"].isna() & co2["change_wrong"].notna()

print("rows where the two disagree:", wrong_rows.sum())
co2[wrong_rows][["country", "year", "co2_total", "change_wrong", "change"]].head(3)

rows where the two disagree: 231


,country,year,co2_total,change_wrong,change
24,Africa Eastern and Southern,2000,425.0290,413.3804,NaN
48,Africa Western and Central,2000,146.5159,-472.8498,NaN
72,Albania,2000,3.2329,-248.4143,NaN


## The pattern behind all of these

Not one of the eight snippets raised an error. They produced:

- a Series where you expected a DataFrame (a),
- a column of `NaN` (b),
- a total that was 30% short (c),
- a table that grew (d),
- a plausible number computed across a boundary (e, and broken snippet 2),
- dates in the wrong order and then in the wrong month (f),
- and 48 rows deleted in silence (broken snippet 1).

The habits that catch all of them are cheap and mechanical:

- **After a merge, compare the row count** before and after, and use `validate=`.
- **After a groupby, check that the parts add up to the whole.**
- **Before `diff`, `ffill`, `shift` or `pct_change`, ask what the rows are a series within** — and
  group by it.
- **Check the dtype of anything that looks like a date** before you sort, filter or group on it.